In [0]:
%pip install python-dotenv

In [0]:
from pyspark.sql.functions import col, current_timestamp, get_json_object
from dotenv import load_dotenv
import os
import time

load_dotenv()

# --- 1. CONFIGURATION (STAYING THE SAME) ---
storage_account_name = "adlsgen2detraining2026"
container_name = "crypto-data-ronit"
storage_account_key = os.getenv("STORAGE_ACCOUNT_KEY")
namespace = 'eventhub-de-training-2026'
event_hub_name = 'capstone-project-coinmarketcap'
conn_str = os.getenv("EVENTHUB_CONNECTION_STRING")

# Configure Storage Access
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", 
    storage_account_key
)

kafka_options = {
    "kafka.bootstrap.servers": "eventhub-de-training-2026.servicebus.windows.net:9093",
    "subscribe": "capstone-project-coinmarketcap",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    # Using kafkashaded prefix is critical for Databricks 
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{conn_str}";',
    # These fix the "Timed out waiting for a node assignment" error
    "kafka.request.timeout.ms": "60000",
    "kafka.session.timeout.ms": "60000",
    "kafka.metadata.max.age.ms": "180000",
    "kafka.group.id": "databricks-cg-ingestion",
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

In [0]:
# --- 2. READ STREAM ---
raw_df = spark.readStream \
    .format("kafka") \
    .options(**kafka_options) \
    .load()

In [0]:
# --- 3. DYNAMIC BRONZE TRANSFORMATION ---
# We use get_json_object to read the "source" field we added to your Python script
bronze_df = raw_df.select(
    col("value").cast("string").alias("raw_content"),
    get_json_object(col("value").cast("string"), "$.source").alias("source"),
    get_json_object(col("value").cast("string"), "$.endpoint_type").alias("endpoint_type"),
    current_timestamp().cast("date").alias("ingestion_date"),
    col("timestamp").alias("event_timestamp")
)

In [0]:
# --- 4. BRONZE LAYER WRITE ---
bronze_storage_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze/raw_market_data"
checkpoint_path_final = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze/_checkpoints/v1"

print("Starting Bronze Ingestion (AvailableNow mode)...")

query = bronze_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path_final) \
    .partitionBy("source", "ingestion_date") \
    .trigger(availableNow=True) \
    .start(bronze_storage_path)

# This is the correct way to wait for a one-time trigger to finish
query.awaitTermination()

print("Bronze write complete! Stream has terminated naturally after processing all data.")

In [0]:
# --- 5. VERIFICATION ---
print("Ingestion complete. Reading back results...")
bronze_results = spark.read.format("delta").load(bronze_storage_path)

# This will show you exactly how many rows came from CMC vs CoinGecko
bronze_results.groupBy("source", "endpoint_type").count().show()

display(bronze_results.sort(col("event_timestamp").desc()))